In [28]:
#Started Sql Analysis

import sqlite3
import pandas as pd

In [29]:
#Creating a Database for our IPL Data
conn = sqlite3.connect("ipl.db")

#Importing Csv Files as dataframe using Pandas
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')


In [30]:
#Cleaning Data 
matches['winner'] = matches['winner'].replace({ 'Kings XI Punjab':'Punjab Kings',
    'Delhi Daredevils':'Delhi Capitals',
    'Deccan Chargers':'Sunrisers Hyderabad',
    'Rising Pune Supergiant':'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'})

matches['season'] = matches['season'].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'})

matches['venue'] = matches['venue'].str.split(',').str[0]
matches['venue'] = matches['venue'].replace({'M.Chinnaswamy Stadium':'M Chinnaswamy Stadium',
    'Punjab Cricket Association Stadium':'Maharaja Yadavindra Singh International Cricket Stadium',
    'Feroz Shah Kotla':'Arun Jaitley Stadium',
    'Sardar Patel Stadium':'Narendra Modi Stadium',
    'Zayed Cricket Stadium':'Sheikh Zayed Stadium'
    })


#Load Data into Sql Database

matches.to_sql('matches',conn,if_exists = 'replace',index = False)
deliveries.to_sql('deliveries',conn,if_exists = 'replace',index = False)



260920

## SQL Queries


In [31]:
#Finding Top 5 Teams with respect to total number of matches won

top_teams = pd.read_sql("""SELECT winner, COUNT(*) as total_wins
FROM matches
GROUP BY winner
ORDER BY total_wins DESC
LIMIT 5""",conn)
top_teams

,winner,total_wins
0,Mumbai Indians,144
1,Chennai Super Kings,138
2,Kolkata Knight Riders,131
3,Royal Challengers Bengaluru,123
4,Sunrisers Hyderabad,117


In [32]:
#Finding Top Batsman with respect to runs scored

top_bat = pd.read_sql("""Select batter,sum(batsman_runs) as total_runs
FROM deliveries
GROUP BY batter
ORDER BY total_runs DESC
limit 10""",conn)
top_bat

,batter,total_runs
0,V Kohli,8014
1,S Dhawan,6769
2,RG Sharma,6630
3,DA Warner,6567
4,SK Raina,5536
5,MS Dhoni,5243
6,AB de Villiers,5181
7,CH Gayle,4997
8,RV Uthappa,4954
9,KD Karthik,4843


In [33]:
#Finding Top Bowler with respect to Wickets Taken

top_bowler = pd.read_sql("""Select bowler,sum(is_wicket) as total_wickets
FROM deliveries
GROUP BY bowler
ORDER BY total_wickets DESC
limit 10""",conn)
top_bowler

,bowler,total_wickets
0,YS Chahal,213
1,DJ Bravo,207
2,PP Chawla,201
3,SP Narine,200
4,R Ashwin,198
5,B Kumar,195
6,SL Malinga,188
7,A Mishra,183
8,JJ Bumrah,182
9,RA Jadeja,169


In [34]:
#Finding Average Runs Scored per Season
avg_run_seasons = pd.read_sql("""SELECT season, AVG(match_runs) as avg_runs_per_match
FROM (
    SELECT matches.id, season, SUM(total_runs) as match_runs
    FROM matches
    JOIN deliveries ON matches.id = deliveries.match_id
    GROUP BY matches.id, season
) 
GROUP BY season
ORDER BY season
""",conn)

avg_run_seasons

,season,avg_runs_per_match
0,2008,309.258621
1,2009,286.894737
2,2010,314.716667
3,2011,289.780822
4,2012,303.418919
5,2013,297.394737
6,2014,315.516667
7,2015,311.067797
8,2016,314.366667
9,2017,318.406780


In [36]:
#Finding Average Runs Scored per venue and Finding Venue
avg_run_venue = pd.read_sql("""SELECT venue, AVG(match_runs) as avg_runs_per_match
FROM (
    SELECT matches.id, venue, SUM(total_runs) as match_runs
    FROM matches
    JOIN deliveries ON matches.id = deliveries.match_id
    GROUP BY matches.id, venue
) 
GROUP BY venue
ORDER BY avg_runs_per_match DESC
""",conn)

avg_run_venue

,venue,avg_runs_per_match
0,Brabourne Stadium,344.629630
1,Punjab Cricket Association IS Bindra Stadium,341.423077
2,Barsapara Cricket Stadium,339.666667
3,Himachal Pradesh Cricket Association Stadium,339.384615
4,Saurashtra Cricket Association Stadium,333.300000
5,Narendra Modi Stadium,331.027778
6,Wankhede Stadium,330.457627
7,M Chinnaswamy Stadium,326.723404
8,Barabati Stadium,325.428571
9,Green Park,324.500000
